In [0]:
# =============================================================================
# ICEBASE — PHASE 3 | NOTEBOOK 02
# Gold Layer — game_revenue
# Idaho Mashers Hockey Analytics Platform
# =============================================================================
# STANDALONE NOTEBOOK — Run attached to icebase-dev cluster.
# Reads from Silver, writes to icebase.gold.game_revenue.
#
# WHAT THIS NOTEBOOK DOES:
#   Builds a per-game revenue summary — one row per game, aggregating
#   all ticket and promo activity for that game. This is the primary
#   table for the Revenue Operations Dashboard in Phase 5.
#
#   Every number here answers a VP of Revenue question:
#     "Which games made us the most money?"
#     "When did promo discounts hurt us most?"
#     "Did Jersey Night actually move the needle on revenue?"
#     "What was our fill rate trend across the season?"
#
# TABLE WRITTEN:
#   icebase.gold.game_revenue
# =============================================================================

In [0]:
# COMMAND ----------
# ── CELL 1: Imports and Config ─────────────────────────────────────────────
 
from pyspark.sql import functions as F
from pyspark.sql.window import Window
 
CATALOG = "icebase"
GOLD    = f"{CATALOG}.gold"
SILVER  = f"{CATALOG}.silver"
 
print("✅ game_revenue notebook loaded")
print(f"   Writing to: {GOLD}.game_revenue")

In [0]:
# COMMAND ----------
# ── CELL 2: Build game_revenue ─────────────────────────────────────────────
 
fact  = spark.table(f"{SILVER}.fact_tickets")
dim_g = spark.table(f"{SILVER}.dim_game")
promo = spark.table(f"{SILVER}.bridge_promo")
 
# ── Per-game ticket aggregations ──────────────────────────────────────────
ticket_agg = (
    fact
    .groupBy("game_id")
    .agg(
        # Revenue
        F.round(F.sum("ticket_price"), 2)           .alias("gross_revenue"),
        F.round(F.avg("ticket_price"), 2)           .alias("avg_ticket_price"),
        F.round(F.max("ticket_price"), 2)           .alias("max_ticket_price"),
        F.round(F.min("ticket_price"), 2)           .alias("min_ticket_price"),
 
        # Volume
        F.count("ticket_id")                        .alias("tickets_sold"),
        F.countDistinct("customer_id")              .alias("unique_buyers"),
 
        # Promo breakdown
        F.sum(F.when(F.col("is_promo_ticket"), 1).otherwise(0))
                                                    .alias("promo_tickets"),
        F.round(
            F.sum(F.when(F.col("is_promo_ticket"), 1).otherwise(0))
            / F.count("ticket_id"), 4
        )                                           .alias("promo_rate"),
 
        # Purchase behavior
        F.round(F.avg("days_before_game"), 1)       .alias("avg_days_before_game"),
        F.sum(F.when(F.col("is_advance_purchase"), 1).otherwise(0))
                                                    .alias("advance_purchase_count"),
 
        # Special event flags
        F.max(F.when(F.col("is_lapsed_reactivation"), 1).otherwise(0)).cast("boolean")
                                                    .alias("had_lapsed_reactivation"),
        F.sum(F.when(F.col("is_lapsed_reactivation"), 1).otherwise(0))
                                                    .alias("lapsed_reactivation_count"),
 
        # Section tier mix — what pct bought premium seats?
        F.round(
            F.sum(F.when(F.col("seat_tier_rank") >= 4, 1).otherwise(0))
            / F.count("ticket_id"), 4
        )                                           .alias("premium_seat_rate"),
    )
)
 
# ── Per-game promo discount totals ────────────────────────────────────────
promo_agg = (
    promo
    .groupBy("game_id")
    .agg(
        F.round(F.sum("discount_amount"), 2)        .alias("total_discount_given"),
        F.round(F.avg("discount_pct") * 100, 1)    .alias("avg_discount_pct"),
        F.sum(F.when(F.col("discount_tier") == "deep", 1).otherwise(0))
                                                    .alias("deep_discount_count"),
    )
)
 
# ── Join to game dimension and compute final metrics ──────────────────────
game_revenue = (
    dim_g
    .join(ticket_agg, "game_id", "left")
    .join(promo_agg,  "game_id", "left")
    .withColumn("total_discount_given",
        F.coalesce(F.col("total_discount_given"), F.lit(0.0)))
    # Net revenue = gross minus discounts given away
    .withColumn("net_revenue",
        F.round(
            F.coalesce(F.col("gross_revenue"), F.lit(0.0)) -
            F.coalesce(F.col("total_discount_given"), F.lit(0.0)),
            2
        )
    )
    # Fill rate from Silver dim_game — already computed
    # Revenue per seat = gross revenue / arena capacity
    .withColumn("revenue_per_seat",
        F.round(
            F.coalesce(F.col("gross_revenue"), F.lit(0.0)) / F.lit(6200),
            2
        )
    )
    # Season running revenue total (window function)
    .withColumn("cumulative_revenue",
        F.round(
            F.sum(F.coalesce(F.col("gross_revenue"), F.lit(0.0)))
              .over(Window.orderBy("game_number")
                          .rowsBetween(Window.unboundedPreceding, Window.currentRow)),
            2
        )
    )
    # 3-game rolling average fill rate — smooths out noise
    .withColumn("fill_rate_3game_avg",
        F.round(
            F.avg("arena_fill_pct")
              .over(Window.orderBy("game_number")
                          .rowsBetween(-2, Window.currentRow)),
            3
        )
    )
    # Revenue index: how does this game compare to season average?
    # > 1.0 = above average, < 1.0 = below average
    .withColumn("revenue_index",
        F.round(
            F.coalesce(F.col("gross_revenue"), F.lit(0.0)) /
            F.avg(F.coalesce(F.col("gross_revenue"), F.lit(0.0)))
              .over(Window.orderBy(F.lit(1))
                          .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)),
            3
        )
    )
    .withColumn("_gold_refreshed_at", F.current_timestamp())
    .orderBy("game_number")
)

In [0]:
# COMMAND ----------
# ── CELL 3: Write to Gold ──────────────────────────────────────────────────
 
(
    game_revenue
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.game_revenue")
)
 
count = spark.table(f"{GOLD}.game_revenue").count()
print(f"✅ game_revenue written | {count:,} rows")

In [0]:
# COMMAND ----------
# ── CELL 4: Quick Sanity Check ─────────────────────────────────────────────
 
print("── Revenue by season phase ──")
display(
    spark.table(f"{GOLD}.game_revenue")
    .groupBy("season_phase")
    .agg(
        F.count("*")                                .alias("games"),
        F.round(F.sum("gross_revenue"), 2)          .alias("total_gross_revenue"),
        F.round(F.sum("total_discount_given"), 2)   .alias("total_discounts"),
        F.round(F.sum("net_revenue"), 2)            .alias("total_net_revenue"),
        F.round(F.avg("arena_fill_pct") * 100, 1)  .alias("avg_fill_pct"),
        F.round(F.avg("promo_rate") * 100, 1)      .alias("avg_promo_rate_pct"),
    )
    .orderBy(F.col("total_gross_revenue").desc())
)
 
# EXPECTED:
#   jersey_night should show by far the highest gross_revenue per game
#   slump should show the highest total_discounts and avg_promo_rate_pct
#   hot_start should show highest avg_fill_pct (excluding jersey night)